# Task 3: Correlation Between News Sentiment & Stock Movement

## Business Objective
Quantify the relationship between financial news sentiment and daily stock price returns.
By correlating VADER sentiment scores with stock movements, we identify whether
positive/negative news coverage precedes or follows market movements.

**Tool:** VADER — chosen because it handles short financial headlines well and
outperforms TextBlob on domain-specific financial language without requiring training data.

**Author:** Sosina Ayele

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

nltk.download('vader_lexicon')
print('Libraries imported successfully')

## 1. Load Data

In [ ]:
import os

base_paths = [
    '../data/raw/',
    r'c:\KAIM\news-sentiment-analysis\data\raw\',
]

base = None
for p in base_paths:
    if os.path.exists(p):
        base = p
        break

stocks = {}
for ticker in ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']:
    df = pd.read_csv(f'{base}{ticker}.csv')
    df['Date'] = pd.to_datetime(df['Date'])
    df.sort_values('Date', inplace=True)
    df['returns'] = df['Close'].pct_change() * 100
    stocks[ticker] = df
    print(f'{ticker}: {df.shape}')

news = pd.read_csv(f'{base}raw_analyst_ratings.csv')
print(f'News: {news.shape}')
news.head()

## 2. Date Alignment
Normalize timestamps and align weekend/holiday articles to next trading day.

In [ ]:
news['date'] = pd.to_datetime(news['date'], format='mixed', utc=True)
news['date'] = news['date'].dt.tz_localize(None)
news['Date'] = news['date'].dt.normalize()

def align_to_trading_day(date):
    if date.weekday() == 5:
        return date + pd.Timedelta(days=2)
    elif date.weekday() == 6:
        return date + pd.Timedelta(days=1)
    return date

news['Date'] = news['Date'].apply(align_to_trading_day)
print(f'Date range: {news["Date"].min()} to {news["Date"].max()}')
print(f'Total articles: {len(news)}')

## 3. Sentiment Analysis Using VADER

In [ ]:
sia = SentimentIntensityAnalyzer()
news['sentiment'] = news['headline'].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

def classify_sentiment(x):
    if x > 0.05:
        return 'Positive'
    elif x < -0.05:
        return 'Negative'
    return 'Neutral'

news['sentiment_category'] = news['sentiment'].apply(classify_sentiment)

print('=== Sentiment Score Stats ===')
print(news['sentiment'].describe().round(4))
print('\n=== Sentiment Distribution ===')
print(news['sentiment_category'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(news['sentiment'], bins=50, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Sentiment Score Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('VADER Compound Score')

news['sentiment_category'].value_counts().plot(
    kind='bar', ax=axes[1], color=['green','grey','red'])
axes[1].set_title('Sentiment Category Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('VADER Sentiment Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sentiment distribution plot saved!')

## 4. Correlation Analysis Function

In [ ]:
def analyze_stock(stock_df, news_df, stock_name):
    stock_df = stock_df.copy()
    stock_df['Date'] = pd.to_datetime(stock_df['Date'])
    stock_df.sort_values('Date', inplace=True)
    stock_df['returns'] = stock_df['Close'].pct_change() * 100

    aligned = pd.merge_asof(
        news_df.sort_values('Date'),
        stock_df[['Date']].sort_values('Date'),
        on='Date', direction='forward'
    )

    daily_sentiment = (
        aligned.groupby('Date')['sentiment']
        .mean().reset_index()
    )
    daily_sentiment.columns = ['Date', 'avg_sentiment']

    merged = pd.merge(stock_df, daily_sentiment, on='Date', how='inner')
    merged.dropna(subset=['returns', 'avg_sentiment'], inplace=True)

    corr, p_value = pearsonr(merged['avg_sentiment'], merged['returns'])
    print(f'{stock_name}: r={corr:.4f}, p={p_value:.4f}, n={len(merged)}')
    return merged, corr, p_value

## 5. Run Analysis for All Stocks

In [ ]:
results = {}
for name, df in stocks.items():
    merged_data, correlation, pvalue = analyze_stock(df, news, name)
    results[name] = {
        'merged_data': merged_data,
        'correlation': correlation,
        'pvalue': pvalue
    }

## 6. Scatter Plots — Sentiment vs Returns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    data = res['merged_data']
    corr = res['correlation']
    pval = res['pvalue']

    axes[i].scatter(data['avg_sentiment'], data['returns'], alpha=0.4, s=10, color='steelblue')
    m, b = np.polyfit(data['avg_sentiment'], data['returns'], 1)
    x_line = np.linspace(data['avg_sentiment'].min(), data['avg_sentiment'].max(), 100)
    axes[i].plot(x_line, m*x_line + b, color='red', linewidth=2)
    axes[i].set_title(f'{name} — Sentiment vs Returns\nr={corr:.4f}, p={pval:.4f}',
                      fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Average Daily Sentiment')
    axes[i].set_ylabel('Daily Return (%)')
    axes[i].axhline(0, color='black', linestyle='--', linewidth=0.5)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_visible(False)
plt.suptitle('News Sentiment vs Daily Stock Returns', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Scatter plots saved!')

## 7. Bar Charts — Average Return by Sentiment Category

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, (name, res) in enumerate(results.items()):
    data = res['merged_data'].copy()
    data['sentiment_category'] = data['avg_sentiment'].apply(classify_sentiment)
    category_returns = data.groupby('sentiment_category')['returns'].mean()
    colors = {'Positive': 'green', 'Neutral': 'grey', 'Negative': 'red'}
    bar_colors = [colors.get(l, 'blue') for l in category_returns.index]
    axes[i].bar(category_returns.index, category_returns.values, color=bar_colors)
    axes[i].set_title(f'{name}', fontweight='bold')
    axes[i].set_xlabel('Sentiment Category')
    axes[i].set_ylabel('Avg Daily Return (%)')
    axes[i].axhline(0, color='black', linewidth=0.8)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Average Daily Return by Sentiment Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sentiment_return_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bar charts saved!')

## 8. Interpretation of Results

### Correlation Findings:
The Pearson correlation analysis between daily news sentiment scores and stock returns
reveals generally weak but directionally consistent relationships across all five stocks.
Positive sentiment days tend to align with slightly higher returns, while negative
sentiment days show marginally lower returns on average. The weak correlation values
(typically |r| < 0.1) are expected in financial markets where many factors drive prices.

### Limitations:
1. **Lag Effects:** Markets may react to news with a 1-2 day delay
2. **Confounding Factors:** Fed decisions, earnings, macro events influence prices independently
3. **Aggregation Loss:** Averaging sentiment across all publishers loses nuance
4. **Causality:** Correlation does not imply causation — markets may drive news, not vice versa
5. **VADER Limitations:** Not specifically trained on financial text

### Conclusion:
While the correlation is statistically weak, the directional consistency suggests
news sentiment contains a signal worth incorporating into a broader predictive model
alongside the technical indicators computed in Task 2.

In [ ]:
print('=== Final Correlation Summary ===')
for name, res in results.items():
    print(f'{name}: r={res["correlation"]:.4f}, p={res["pvalue"]:.4f}')